# TOPIC 1: Explainable Machine Learning for Gallstone Disease

**Dataset:** UCI Gallstone Disease Dataset (`gallstone-1`)   

## Index of the Notebook

1. **Exploratory Data Analysis (EDA):** Inspect dataset dimensions, feature distributions, missing values, and class balance.
2. **Data Preprocessing:** Separate continuous and binary features, fit `StandardScaler` on continuous features only via `ColumnTransformer`, and perform stratified train/test split ($80/20$).
3. **Model Development & Cross-Validation:** Train 4 classifiers (Logistic Regression, Random Forest, SVM, XGBoost) using 5-Fold Stratified Cross-Validation.
4. **Evaluation:** Compare holdout performance metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC) and plot confusion matrices.
5. **Model & Scaler Export:** Save top-performing model (`best_model.joblib`) and preprocessor (`scaler.joblib`) to the `dataset/` directory.
6. **Explainability Analysis (XAI - SHAP):** Compute SHAP values, plot global feature importance (Beeswarm & Bar plots), and analyze individual predictions (Waterfall plot).

In [ ]:
# NECESSARY LIBRARIES FROM THE COURSE

import os
import warnings
import joblib

# 1.2: Introduction to Python Libraries for Machine Learning
# Data handling, visualization, and preprocessing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & Model Selection
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 3.2.1 & 3.2.5: Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# 3.2.3: Evaluation Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')

In [ ]:
#uploading the dataset from the location of the excel on the project

#dataset route to the excel
data_path = os.path.join('..', 'dataset', 'dataset-uci.xlsx')

#uploading the dataset with calamine
df = pd.read_excel(data_path, engine='calamine')

#printing to check the succes
print(f"Dataset uploaded. Size: {df.shape}")
df.head()

1. **Exploratory Data Analysis (EDA):** Inspect dataset dimensions, feature distributions, and class balance.

In [ ]:
#setting the target
target_col = 'Gallstone Status'

#checking missing values
print(f"Total missing values: {df.isnull().sum().sum()}")

#Check target variable class balance
print("\nTarget Class Distribution")
print(df[target_col].value_counts())

#plotting class balance
plt.figure(figsize=(6, 4))
sns.countplot(x=target_col, data=df, hue=target_col, palette='Set2', legend=False)
plt.title('Target Distribution (0 = Healthy, 1 = Disease)')
plt.xlabel('Gallstone Status')
plt.ylabel('Patient Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

2. **Data Preprocess:** Apply feature scaling and stratified test split ($80/20$).

In [ ]:
#separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

#stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

#feature scaling with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#save fitted scaler for future inference
scaler_path = os.path.join('..', 'dataset', 'scaler.joblib')
joblib.dump(scaler, scaler_path)

print(f"Split complete: {X_train_scaled.shape[0]} train samples, {X_test_scaled.shape[0]} test samples.")
print(f"Scaler saved to: {scaler_path}")

3. **Model Development:** Train 4 classifiers (Logistic Regression, Random Forest, SVM, XGBoost) with stratified cross-validation.

In [ ]:
#dictionary for the 4 models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

#looping the dictionary of models through the data
print("training classification models")
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"{name} trained.")

4. **Evaluation:** Compare performance metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC) and plot confusion matrices.

In [ ]:
#list to store the results
results = []

#looping thorugh the models to seee the different results
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    #we append the results of the 5 calculated metrics into the list to compare trugh agaisnt predictions
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba)
    })

#metrics summary
results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False).reset_index(drop=True)
print("\nModel Performance Summary")
print(results_df.round(4))

#plotting confusion matrix's
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

#looping through the models and plotting the confusion matrix
for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name}')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

#plotting the results of the confusion matrix
plt.tight_layout()
plt.show()

5. **Model Export:** Save the top-performing model (`best_model.joblib`) to the `dataset/` directory for SHAP/LIME explainability analysis.

In [ ]:
#seeing which one is the top-performing model based on the highest F1-Score
best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

#define the output path in dataset/ directory
best_model_path = os.path.join('..', 'dataset', 'best_model.joblib')

#saving model binary using joblib
joblib.dump(best_model, best_model_path)

#printing the results
print(f"\n Best Model: '{best_model_name}' (F1-Score: {results_df.iloc[0]['F1-Score']:.4f})")
print(f"Model successfully exported to: {best_model_path} for SHAP analysis.")